In [1]:
# Stage 1: compute image features and save the result to Google Drive.
# Reads images from Drive, imports feature functions from the GitHub repo,
# and writes one CSV. No figures, so this notebook stays small and reliable.
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

# Source images on Drive.
RAW_RTTS_DIR = "/content/drive/MyDrive/adaptive_perception_data/raw/RTTS"

# Output folder on Drive (permanent, survives runtime resets).
OUTPUT_TABLES_DIR = "/content/drive/MyDrive/perception-difficulty-dve/results/tables"
os.makedirs(OUTPUT_TABLES_DIR, exist_ok=True)

# Code: clone the repo so src/features.py is importable.
REPO_ROOT = "/content/perception-difficulty-dve"
REPO_URL = "https://github.com/aaaraafaat/perception-difficulty-dve.git"
if not os.path.isdir(REPO_ROOT):
    os.system(f"git clone {REPO_URL} {REPO_ROOT}")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import src.features as feat

if not os.path.isdir(RAW_RTTS_DIR):
    raise FileNotFoundError(f"RTTS images not found at: {RAW_RTTS_DIR}")

feature_names = [name for name in dir(feat) if name.endswith("_score")]
print("Drive mounted, repo cloned.")
print(f"Features available: {len(feature_names)}")
for name in feature_names:
    print("  -", name)

Mounted at /content/drive
Drive mounted, repo cloned.
Features available: 8
  - brightness_score
  - colourfulness_score
  - contrast_score
  - dark_channel_score
  - entropy_score
  - noise_score
  - saturation_score
  - sharpness_score


In [2]:
# Compute every feature for every RTTS image and save the table to Drive.
# Resume logic: if the CSV already exists, only missing features are computed.
import cv2
import glob
import pandas as pd

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
csv_path = os.path.join(OUTPUT_TABLES_DIR, "features_RTTS.csv")
available_features = sorted(name for name in dir(feat) if name.endswith("_score"))


def compute_selected_features(image_bgr, feature_names):
    """Return a dict of the named feature scores for one image."""
    return {name: getattr(feat, name)(image_bgr) for name in feature_names}


# --- Step 1: load existing table from Drive, or build the image list --------
if os.path.exists(csv_path):
    features = pd.read_csv(csv_path)
    print(f"Loaded existing table: {len(features)} rows, "
          f"features {[c for c in features.columns if c.endswith('_score')]}")
else:
    image_paths = []
    for extension in IMAGE_EXTENSIONS:
        image_paths += glob.glob(os.path.join(RAW_RTTS_DIR, "**", f"*{extension}"), recursive=True)
        image_paths += glob.glob(os.path.join(RAW_RTTS_DIR, "**", f"*{extension.upper()}"), recursive=True)
    image_paths = sorted(set(image_paths))

    identity_rows = []
    for image_path in image_paths:
        image_bgr = cv2.imread(image_path)
        if image_bgr is None:
            continue  # skip any file that fails to load
        image_height, image_width = image_bgr.shape[:2]
        identity_rows.append({
            "filename": os.path.basename(image_path),
            "relpath": os.path.relpath(image_path, RAW_RTTS_DIR),
            "dataset": "RTTS",
            "width": image_width,
            "height": image_height,
        })
    features = pd.DataFrame(identity_rows)
    print(f"Built new table: {len(features)} images")

# --- Step 2: compute only the features not already present ------------------
missing_features = [name for name in available_features if name not in features.columns]
if not missing_features:
    print("All features already computed.")
else:
    print(f"Computing: {missing_features}")
    computed_rows = []
    for relpath in features["relpath"]:
        image_bgr = cv2.imread(os.path.join(RAW_RTTS_DIR, relpath))
        computed_rows.append(compute_selected_features(image_bgr, missing_features))
    features = pd.concat([features, pd.DataFrame(computed_rows, index=features.index)], axis=1)
    print(f"Computed {len(missing_features)} feature(s) for {len(features)} images.")

# --- Step 3: save to Drive --------------------------------------------------
features.to_csv(csv_path, index=False)
print("Saved to Drive:", csv_path)

# --- Step 4: text summary ---------------------------------------------------
feature_columns = [c for c in features.columns if c.endswith("_score")]
print(f"\n{len(features)} images, {len(feature_columns)} features")
print(features[feature_columns].describe().round(3).to_string())

Loaded existing table: 4322 rows, features ['brightness_score', 'colourfulness_score', 'contrast_score', 'dark_channel_score', 'entropy_score', 'noise_score', 'saturation_score', 'sharpness_score']
All features already computed.
Saved to Drive: /content/drive/MyDrive/perception-difficulty-dve/results/tables/features_RTTS.csv

4322 images, 8 features
       brightness_score  colourfulness_score  contrast_score  dark_channel_score  entropy_score  noise_score  saturation_score  sharpness_score
count          4322.000             4322.000        4322.000            4322.000       4322.000     4322.000          4322.000         4322.000
mean              0.575                0.104           0.385               0.452          6.941        0.070             0.115            0.243
std               0.111                0.077           0.135               0.127          0.588        0.057             0.108            0.280
min               0.107                0.000           0.062            

In [3]:
pd.read_csv(csv_path)

,filename,relpath,dataset,width,height,brightness_score,colourfulness_score,contrast_score,dark_channel_score,entropy_score,noise_score,saturation_score,sharpness_score
0,AM_Bing_211.png,JPEGImages/AM_Bing_211.png,RTTS,478,270,0.490792,0.066641,0.489184,0.290129,7.314641,0.226984,0.097319,1.000000
1,AM_Bing_217.png,JPEGImages/AM_Bing_217.png,RTTS,990,576,0.653333,0.065291,0.358655,0.407105,7.459955,0.413905,0.044910,1.000000
2,AM_Bing_222.png,JPEGImages/AM_Bing_222.png,RTTS,480,320,0.595887,0.061021,0.376517,0.438060,7.276953,0.127120,0.045474,0.412778
3,AM_Bing_229.png,JPEGImages/AM_Bing_229.png,RTTS,600,474,0.557399,0.138100,0.525717,0.425849,7.490536,0.110936,0.087239,0.493579
4,AM_Bing_232.png,JPEGImages/AM_Bing_232.png,RTTS,619,398,0.489609,0.185628,0.523380,0.124316,7.901619,0.432952,0.215502,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4317,hv9_215.png,JPEGImages/hv9_215.png,RTTS,956,642,0.593959,0.049795,0.239125,0.537546,6.121815,0.006982,0.078866,0.001327
4318,hv9_221.png,JPEGImages/hv9_221.png,RTTS,957,645,0.579871,0.051171,0.242601,0.523660,6.273078,0.008326,0.076617,0.001960
4319,hv9_223.png,JPEGImages/hv9_223.png,RTTS,956,647,0.569044,0.048237,0.290981,0.517740,6.878369,0.010511,0.060110,0.004196
4320,hv9_36.png,JPEGImages/hv9_36.png,RTTS,738,455,0.528418,0.079625,0.622940,0.437607,7.671793,0.032759,0.132958,0.054255
